In [1]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import os 
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage , HumanMessage
import gradio as gr

d:\project\Ai research\Ai-Research-Assistant\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def read_pdf(pdf):
    text = PdfReader(pdf)
    full_text= ""
    for doc in text.pages:
        full_text += doc.extract_text()+'\n'

    return full_text    

In [3]:
pdf_path = 'Modern Application Design and Development (CS-4103)_B.pdf'
data = read_pdf(pdf_path)
print(data)

The University of Azad Jammu and Kashmir Muzaffarabad
Print Date: 11-Mar-2026
Subject Award List
Course Title :
Instructor:Semester No :Department/Institute:
Course Credits :
No Of Students:Program Session :Faculty:
Modern Application Design and Development (CS-4103)
Dr. Adil Aslam MirFaculty Of Science
BS(CS) ( 2022-2026 ) (Morning-B) (Fall)Department Of Computer Sciences and IT
7 (Fall 2025)
3(3+0)
 48 
 Marks Obtained
Reg No NameQuiz Sessional TerminalObtain Grad GP Roll No Marks In Words %age( 15.00  ) ( 45.00 ) ( 75.00 )Total( 15.00 )Assignments
2022-UMDB-002764 Fatima Abbasi  13.00  9.00  32.00  60.00  114.00 B+  3.70  61 One Hundred Fourteen  150  76.0
2022-UMDB-002765 Muhammad Sufyan Chishty  13.00  8.00  27.00  41.00  89.00 C  2.60  62 Eighty-Nine  150  59.3
2022-UMDB-002766 Syeda Renad Masroor  13.00  8.00  31.00  49.00  101.00 B  3.10  63 One Hundred One  150  67.3
2022-UMDB-002767 Usman Tariq  9.00  14.00  21.00  34.00  78.00 C  2.10  64 Seventy-Eight  150  52.0
2022-UMDB-0

In [4]:
spliter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
chunks = spliter.create_documents([data])


In [5]:
print(len(chunks))

15


In [8]:
#Here is permenant storage for the vectors, they store vector letter at time of query we just fetch vector form there not agin converting
vector = 'vector.db'

In [6]:
#Turing these chunks into vectors 
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
#vector_store = Chroma.from_documents(documents=chunks,embedding=embeddings,persist_directory=vector)
#print(f'Vectorstore created with {vector_store._collection.count()} documents')

d:\project\Ai research\Ai-Research-Assistant\venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Now Time to combined the both LLm and retrival

In [9]:
vector_store = Chroma(embedding_function=embeddings,persist_directory=vector)
print(f'Vectorstore created with {vector_store._collection.count()} documents')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vectorstore created with 15 documents


In [10]:
retrival = vector_store.as_retriever()

In [11]:
retrival.invoke('How is mir ikram ')

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Document(page_content='2022-UMDB-002949 Najm Ud Din Yabgo  13.00  9.00  25.00  32.00  79.00 C  2.10  110 Seventy-Nine  150  52.7\n2022-UMDB-002800 Raja Aqib Sarwar  14.00  8.00  31.00  34.00  87.00 C  2.50  111 Eighty-Seven  150  58.0\n2022-UMDB-002801 Mir Ikramullah Raesani  14.00  10.00  29.00  46.00  99.00 B  3.00  113 Ninety-Nine  150  66.0\n2022-UMDB-002802 Wasiq Sardar  13.00  9.00  29.00  41.00  92.00 C  2.70  114 Ninety-Two  150  61.3'),
 Document(page_content='2022-UMDB-002779 Nasir  Mahmood  13.00  9.00  33.00  39.00  94.00 C  2.80  84 Ninety-Four  150  62.7\n2022-UMDB-002780 Ibrahim Zakir  12.00  14.00  26.00  27.00  79.00 C  2.10  85 Seventy-Nine  150  52.7\n2022-UMDB-002783 Tooba Saleem  13.00  8.50  31.00  24.00  77.00 C  2.00  88 Seventy-Seven  150  51.3\n2022-UMDB-002976 Omar Iqbal  13.00  8.00  35.00  42.00  98.00 B  3.00  89 Ninety-Eight  150  65.3'),
 Document(page_content='2022-UMDB-002787 Ahmed Mursaleen  14.00  13.00  26.00  37.00  90.00 C  2.60  95 Ninety  150  

In [12]:
load_dotenv()

True

In [13]:
llm = ChatOpenAI(base_url='https://generativelanguage.googleapis.com/v1beta/openai/',api_key=os.getenv('GEMINI_API_KEY'),model='gemini-2.5-flash')

In [14]:
SYSTEM_PROMPT = """
You are expert student assistent , You task is to give result to student based on the name,
if the student name existed in the your context then give them answer ,with all of info of that student , 
if they not exited then asked this student cannot found
here is {context}"""

In [15]:
def question_anser(question,history):
    text = retrival.invoke(question)
    context = '\n\n'.join(doc.page_content for doc in text)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    response = llm.invoke([SystemMessage(system_prompt),HumanMessage(question)])
    return response.content


In [17]:
question_anser('Give me data of Mir Ikrammullah Reasani',{})

'I found the data for **Mir Ikramullah Raesani**:\n\n**Student ID:** 2022-UMDB-002801\n**Name:** Mir Ikramullah Raesani\n**Marks Obtained:** 14.00, 10.00, 29.00, 46.00\n**Total Marks:** 99.00\n**Grade:** B\n**GPA:** 3.00\n**Rank/Order:** 113\n**Marks in Words:** Ninety-Nine\n**Maximum Marks:** 150\n**Percentage:** 66.0'